---
## Lista de funciones avanzada

La forma directa es generar una lista y pasar las funciones al metodo `.agg()`



In [5]:
import pandas as pd

df = pd.DataFrame({
    "categoria": ["Portátiles","Portátiles","Portátiles","Monitores","Monitores","Periféricos","Periféricos","Periféricos","Periféricos"],
    "importe":   [899, 899, 799, 349, 349, 45, 25, 45, 35],
    "unidades":  [1, 2, 1, 1, 3, 5, 2, 4, 1]
})
# Cada funcion equivale a una columna, cada categoría tiene su resultado calculado
lista_funciones = df.groupby("categoria")["importe"].agg(["mean", "sum", "count", "min", "max"])
print('Una columna por función: ')
print(lista_funciones)

Una columna por función: 
                   mean   sum  count  min  max
categoria                                     
Monitores    349.000000   698      2  349  349
Periféricos   37.500000   150      4   25   45
Portátiles   865.666667  2597      3  799  899


---
# **.agg()** con diccionario

Funciones distintas por columna, cuando se quieren aplicar funciones diferentes a columnas diversas, se pasa un diccionario donde las claves son los nombres de columna y los valores son las funciones,

In [8]:
resultado = df.groupby("categoria").agg({ "importe":  ["mean", "sum"],
    "unidades": ["sum", "max"]})
print(resultado)

                importe       unidades    
                   mean   sum      sum max
categoria                                 
Monitores    349.000000   698        4   3
Periféricos   37.500000   150       12   5
Portátiles   865.666667  2597        4   2


---
# El resultado tiene un **Multindex** en las columnas

El primer nivel es el nombre de la columna original; la segunda de la función aplicada.
Esto puede dificultar el acceso posterior a las columnas. La solución es aplanar el `MultiIndex`.

In [ ]:
#El multindez tiene un resultado real solamente la primera vez que se ejecuta...

#resultado.columns = ['_'.join(col) for col in resultado.columns]
#resultado = resultado.reset_index()
#print(resultado)

---
# Agregaciones nombradas - **forma más limpia**

La forma mas profesional y legible de usar `.agg()` es con agregaciones nombradas. Se pasan tuplas donde el primer elemento es el nombre deseado para la columna resultado y el segundo es la función.

In [9]:
resultado = df.groupby("categoria")["importe"].agg(
    importe_medio=("mean"),
    renueve_total=("sum"),
    numero_ventas=("count"),
    precio_minimo=("min"),
    precio_maximo=("max")
).reset_index().sort_values("renueve_total", ascending=False)
print(resultado)

     categoria  importe_medio  renueve_total  numero_ventas  precio_minimo  \
2   Portátiles     865.666667           2597              3            799   
0    Monitores     349.000000            698              2            349   
1  Periféricos      37.500000            150              4             25   

   precio_maximo  
2            899  
0            349  
1             45  


---
# Groupby por dos columnas con múltiples métricas

La combinación de groupby por dos columnas con el `.agg()` produce el análisis mas completo. Cada combinación única de valores en ambas columnas forma un grupo independiente.

In [12]:
df["ciudad"] = ["Madrid","Madrid","Barcelona","Madrid","Barcelona","Madrid","Sevilla","Barcelona","Sevilla"]

resultado = (
    df
    .groupby(["categoria", "ciudad"])
    .agg(
        num_ventas    = ("importe", "count"),
        revenue_total = ("importe", "sum"),
        ticket_medio  = ("importe", "mean")
    )
    .reset_index()
    .sort_values(["categoria", "revenue_total"], ascending=[True, False])
)

print(resultado)

     categoria     ciudad  num_ventas  revenue_total  ticket_medio
0    Monitores  Barcelona           1            349         349.0
1    Monitores     Madrid           1            349         349.0
4  Periféricos    Sevilla           2             60          30.0
2  Periféricos  Barcelona           1             45          45.0
3  Periféricos     Madrid           1             45          45.0
6   Portátiles     Madrid           2           1798         899.0
5   Portátiles  Barcelona           1            799         799.0
